In [1]:
# ============================================================
# CELL 1 — Import Libraries & Setup Database
# ============================================================

import pandas as pd
import sqlite3
import os

# Load the cleaned dataset
file_path = '/Users/aishwaryadeshwal/Desktop/customer-churn-prediction/data/processed/churn_cleaned.csv'
df = pd.read_csv(file_path)

print("✅ Dataset loaded!")
print(f"📊 Shape: {df.shape}")

# Create SQLite database
db_path = '/Users/aishwaryadeshwal/Desktop/customer-churn-prediction/sql/churn_database.db'
conn = sqlite3.connect(db_path)

# Save dataframe as SQL table
df.to_sql('customers', conn, if_exists='replace', index=False)

print("✅ Database created!")
print("✅ Table 'customers' created with all data!")
print(f"\n📁 Database saved at: sql/churn_database.db")

✅ Dataset loaded!
📊 Shape: (7043, 20)
✅ Database created!
✅ Table 'customers' created with all data!

📁 Database saved at: sql/churn_database.db


In [2]:
# ============================================================
# CELL 2 — Overall Churn Rate
# ============================================================

query = """
SELECT 
    Churn,
    COUNT(*) as total_customers,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as percentage
FROM customers
GROUP BY Churn
"""

result = pd.read_sql_query(query, conn)
print("📊 OVERALL CHURN RATE")
print("=" * 40)
print(result.to_string(index=False))

📊 OVERALL CHURN RATE
Churn  total_customers  percentage
   No             5174       73.46
  Yes             1869       26.54


In [3]:
# ============================================================
# CELL 3 — Churn by Contract Type
# ============================================================

query = """
SELECT 
    Contract,
    COUNT(*) as total_customers,
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) as churned,
    SUM(CASE WHEN Churn = 'No' THEN 1 ELSE 0 END) as retained,
    ROUND(SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as churn_rate
FROM customers
GROUP BY Contract
ORDER BY churn_rate DESC
"""

result = pd.read_sql_query(query, conn)
print("📊 CHURN BY CONTRACT TYPE")
print("=" * 40)
print(result.to_string(index=False))

📊 CHURN BY CONTRACT TYPE
      Contract  total_customers  churned  retained  churn_rate
Month-to-month             3875     1655      2220       42.71
      One year             1473      166      1307       11.27
      Two year             1695       48      1647        2.83


In [4]:
# ============================================================
# CELL 4 — High Risk Customers
# ============================================================

query = """
SELECT 
    Contract,
    InternetService,
    PaymentMethod,
    COUNT(*) as total_customers,
    ROUND(AVG(MonthlyCharges), 2) as avg_monthly_charges,
    ROUND(SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as churn_rate
FROM customers
WHERE Contract = 'Month-to-month'
  AND InternetService = 'Fiber optic'
GROUP BY PaymentMethod
ORDER BY churn_rate DESC
"""

result = pd.read_sql_query(query, conn)
print("📊 HIGH RISK CUSTOMER SEGMENTS")
print("=" * 40)
print(result.to_string(index=False))

📊 HIGH RISK CUSTOMER SEGMENTS
      Contract InternetService             PaymentMethod  total_customers  avg_monthly_charges  churn_rate
Month-to-month     Fiber optic          Electronic check             1307                87.18       60.37
Month-to-month     Fiber optic              Mailed check              201                82.59       50.75
Month-to-month     Fiber optic Bank transfer (automatic)              327                88.25       45.57
Month-to-month     Fiber optic   Credit card (automatic)              293                87.97       41.64


In [5]:
# ============================================================
# CELL 5 — Revenue Impact Analysis
# ============================================================

query = """
SELECT 
    Churn,
    COUNT(*) as total_customers,
    ROUND(AVG(MonthlyCharges), 2) as avg_monthly_charges,
    ROUND(SUM(MonthlyCharges), 2) as total_monthly_revenue,
    ROUND(AVG(tenure), 1) as avg_tenure_months
FROM customers
GROUP BY Churn
"""

result = pd.read_sql_query(query, conn)
print("📊 REVENUE IMPACT ANALYSIS")
print("=" * 40)
print(result.to_string(index=False))

# Calculate revenue lost
query2 = """
SELECT 
    ROUND(SUM(MonthlyCharges), 2) as monthly_revenue_lost
FROM customers
WHERE Churn = 'Yes'
"""
lost = pd.read_sql_query(query2, conn)
print(f"\n💸 Monthly Revenue Lost to Churn: ${lost['monthly_revenue_lost'][0]:,.2f}")
print(f"💸 Annual Revenue Lost to Churn:  ${lost['monthly_revenue_lost'][0]*12:,.2f}")

📊 REVENUE IMPACT ANALYSIS
Churn  total_customers  avg_monthly_charges  total_monthly_revenue  avg_tenure_months
   No             5174                61.27              316985.75               37.6
  Yes             1869                74.44              139130.85               18.0

💸 Monthly Revenue Lost to Churn: $139,130.85
💸 Annual Revenue Lost to Churn:  $1,669,570.20


In [6]:
# ============================================================
# CELL 6 — Customer Lifetime Value by Segment
# ============================================================

query = """
SELECT 
    Contract,
    COUNT(*) as total_customers,
    ROUND(AVG(tenure), 1) as avg_tenure_months,
    ROUND(AVG(MonthlyCharges), 2) as avg_monthly_charges,
    ROUND(AVG(tenure * MonthlyCharges), 2) as avg_lifetime_value
FROM customers
GROUP BY Contract
ORDER BY avg_lifetime_value DESC
"""

result = pd.read_sql_query(query, conn)
print("📊 CUSTOMER LIFETIME VALUE BY CONTRACT")
print("=" * 40)
print(result.to_string(index=False))

📊 CUSTOMER LIFETIME VALUE BY CONTRACT
      Contract  total_customers  avg_tenure_months  avg_monthly_charges  avg_lifetime_value
      Two year             1695               56.7                60.77             3706.76
      One year             1473               42.0                65.05             3029.83
Month-to-month             3875               18.0                66.40             1370.12


In [7]:
# ============================================================
# CELL 7 — Save All Queries to sql folder
# ============================================================

sql_queries = """
-- ================================================
-- Customer Churn Analysis Queries
-- ================================================

-- 1. Overall Churn Rate
SELECT 
    Churn,
    COUNT(*) as total_customers,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as percentage
FROM customers
GROUP BY Churn;

-- 2. Churn by Contract Type
SELECT 
    Contract,
    COUNT(*) as total_customers,
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) as churned,
    ROUND(SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as churn_rate
FROM customers
GROUP BY Contract
ORDER BY churn_rate DESC;

-- 3. High Risk Segments
SELECT 
    Contract,
    InternetService,
    PaymentMethod,
    COUNT(*) as total_customers,
    ROUND(SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as churn_rate
FROM customers
WHERE Contract = 'Month-to-month'
  AND InternetService = 'Fiber optic'
GROUP BY PaymentMethod
ORDER BY churn_rate DESC;

-- 4. Revenue Lost to Churn
SELECT 
    ROUND(SUM(MonthlyCharges), 2) as monthly_revenue_lost,
    ROUND(SUM(MonthlyCharges) * 12, 2) as annual_revenue_lost
FROM customers
WHERE Churn = 'Yes';

-- 5. Customer Lifetime Value
SELECT 
    Contract,
    ROUND(AVG(tenure * MonthlyCharges), 2) as avg_lifetime_value
FROM customers
GROUP BY Contract
ORDER BY avg_lifetime_value DESC;
"""

save_path = '/Users/aishwaryadeshwal/Desktop/customer-churn-prediction/sql/analysis_queries.sql'
with open(save_path, 'w') as f:
    f.write(sql_queries)

print("✅ SQL queries saved to sql/analysis_queries.sql")
print("✅ SQL Analysis Phase COMPLETE!")
print("\n🎯 Next Phase: Machine Learning Pipeline")

✅ SQL queries saved to sql/analysis_queries.sql
✅ SQL Analysis Phase COMPLETE!

🎯 Next Phase: Machine Learning Pipeline
